# Descripción de las fuentes de datos

Inventario y contexto de las fuentes de datos del proyecto. Este notebook documenta qué datos tenemos y qué representan, antes de perfilarlos o transformarlos.

## 1. Contexto de recepción del dataset

Los datos de trabajo fueron entregados a los participantes del bootcamp de Ingeniería de Datos mediante correo electrónico el 15 de julio de 2026 a las 11:11, por el ingeniero David Vargas Maida.

El paquete incluía:

- Un archivo ZIP con exports CSV de tres dominios: university/, billing/ y crm/.
- README.md — instrucciones y alcance del proyecto.
- manifest.json — catálogo de esquema (tablas, columnas y conteos esperados).

En el repositorio, los CSV se ubican en data/raw/ organizados por dominio. La entrega por correo describe la procedencia del paquete del bootcamp; los tres dominios simulan sistemas de origen distintos dentro del escenario de negocio.

## 2. Fuentes de negocio (simuladas)

Los datos representan tres sistemas independientes sobre la misma organización: una institución que gestiona formación académica, facturación de suscripciones y relaciones comerciales (CRM).

| Dominio | Sistema simulado | Tablas | Filas totales |
|---------|------------------|--------|---------------|
| University | Gestión académica | 6 | 90 508 |
| Billing | Facturación y pagos | 6 | 305 000 |
| CRM | Ventas y relación con clientes | 6 | 52 000 |

Total: 18 tablas, 447 508 filas. Formato de origen: CSV (delimitador `,`, encabezado en primera fila).

## 3. Dominio University

Datos del área académica: personas, cursos, inscripciones y calificaciones.

| Archivo | Filas | Descripción |
|---------|-------|-------------|
| semesters.csv | 8 | Periodos académicos |
| professors.csv | 200 | Profesores |
| students.csv | 5 000 | Estudiantes  |
| courses.csv | 300 | Catálogo de cursos, vinculados a un profesor |
| enrollments.csv | 25 000 | Inscripciones de estudiantes a cursos por semestre |
| grades.csv | 60 000 | Calificaciones por inscripción |


## 4. Dominio Billing

Datos del área de facturación: clientes, productos, suscripciones, facturas y pagos.

| Archivo | Filas | Descripción |
|---------|-------|-------------|
| customers.csv | 10 000 | Clientes |
| products.csv | 200 | Productos/planes facturables |
| subscriptions.csv | 15 000 | Suscripciones de clientes a productos |
| invoices.csv | 50 000 | Facturas emitidas |
| invoice_items.csv | 150 000 | Líneas de detalle por factura |
| payments.csv | 80 000 | Pagos asociados a facturas |


## 5. Dominio CRM

Datos del área comercial: cuentas, contactos, pipeline de ventas y actividades.

| Archivo | Filas | Descripción |
|---------|-------|-------------|
| accounts.csv | 5 000 | Cuentas (empresas u organizaciones) |
| contacts.csv | 15 000 | Contactos vinculados a una cuenta |
| leads.csv | 2 000 | Leads comerciales |
| opportunities.csv | 3 000 | Oportunidades de venta por cuenta |
| opportunity_contacts.csv | 6 000 | Relación N:N entre oportunidades y contactos |
| activities.csv | 20 000 | Actividades (llamadas, reuniones, etc.) sobre contactos / oportunidades |


## 6. Metadatos de referencia

| Archivo | Rol |
|---------|-----|
| manifest.json | Catálogo oficial: nombre de tablas, columnas y conteos esperados por fila. Referencia para validar ingesta. |
| README.md | Guía del proyecto: objetivos, fases, stack y criterios de evaluación. No es una fuente de datos de negocio. |

El campo path dentro de manifest.json apunta al directorio del generador de datos, no a este repositorio. Usar solo las claves rows y cols como referencia de esquema.

## 7. Catálogo de fuentes

La celda siguiente construye un inventario a partir de manifest.json y verifica que los CSV existan en data/raw/.

In [7]:
import json
from pathlib import Path
import pandas as pd


PROJECT_ROOT = Path("/home/jovyan/work")
RAW_PATH = PROJECT_ROOT / "data" / "raw"
MANIFEST_PATH = PROJECT_ROOT / "manifest.json"

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

rows = []
for domain, tables in manifest["domains"].items():
    for table, meta in tables.items():
        csv_path = RAW_PATH / domain / f"{table}.csv"
        rows.append(
            {
                "dominio": domain,
                "tabla": table,
                "archivo": f"{table}.csv",
                "filas_esperadas": meta["rows"],
                "columnas": len(meta["cols"]),
                "csv_presente": csv_path.exists(),
            }
        )

catalog = pd.DataFrame(rows).sort_values(["dominio", "tabla"]).reset_index(drop=True)

print(f"Proyecto:       {manifest['project']}")
print(f"Tablas:         {len(catalog)}")
print(f"Filas totales:  {catalog['filas_esperadas'].sum():,}")
print(f"CSV faltantes:  {(~catalog['csv_presente']).sum()}")
print()
print('Catalogo de archivos CSV (desde manifest.json)')
catalog

Proyecto:       crm-billing-universidad
Tablas:         18
Filas totales:  446,708
CSV faltantes:  0

Catalogo de archivos CSV (desde manifest.json)


,dominio,tabla,archivo,filas_esperadas,columnas,csv_presente
0,billing,customers,customers.csv,10000,8,True
1,billing,invoice_items,invoice_items.csv,150000,6,True
2,billing,invoices,invoices.csv,50000,7,True
3,billing,payments,payments.csv,80000,5,True
4,billing,products,products.csv,200,6,True
5,billing,subscriptions,subscriptions.csv,15000,6,True
6,crm,accounts,accounts.csv,5000,7,True
7,crm,activities,activities.csv,20000,6,True
8,crm,contacts,contacts.csv,15000,8,True
9,crm,leads,leads.csv,2000,8,True


# Esquema de columnas por tabla (desde manifest.json)

In [9]:
for domain, tables in manifest["domains"].items():
    print(f"\n ___{domain.upper()}___ ")
    for table, meta in tables.items():
        cols = ", ".join(meta["cols"])
        print(f"\n  {table} ({meta['rows']:,} filas):\n     {cols}")


 ___UNIVERSITY___ 

  semesters (8 filas):
     semester_id, code, year, half, start_date, end_date

  professors (200 filas):
     professor_id, first_name, last_name, email, department, hired_at

  students (5,000 filas):
     student_id, first_name, last_name, email, birth_date, enrolled_at, country

  courses (300 filas):
     course_id, code, name, credits, department, professor_id

  enrollments (25,000 filas):
     enrollment_id, enrolled_at, status, student_id, course_id, semester_id

  grades (60,000 filas):
     grade_id, assessment, score, weight, graded_at, enrollment_id

 ___BILLING___ 

  customers (10,000 filas):
     customer_id, external_ref, first_name, last_name, email, country, created_at, segment

  products (200 filas):
     product_id, sku, name, category, monthly_price, active

  subscriptions (15,000 filas):
     subscription_id, status, start_date, end_date, customer_id, product_id

  invoices (50,000 filas):
     invoice_id, issued_at, due_at, total, status,